# Stock Market Flask API

This project is a Flask-based API that retrieves company information, stock market data, historical data, and analytical insights using the Yahoo Finance API through the `yfinance` Python library.

## Features

- Company information endpoint
- Real-time stock market data endpoint
- Historical stock price data endpoint
- Analytical insights endpoint
- JSON-based API responses

## Technologies Used

- Python
- Flask
- yfinance
- Pandas
- VS Code


In [2]:
from flask import Flask, jsonify, request
import yfinance as yf

# Create Flask app
app = Flask(__name__)

# Home route
@app.route("/")
def home():
    return jsonify({"message": "Stock API is running"})


# Company Information Endpoint
@app.route("/company-info/<symbol>", methods=["GET"])
def company_info(symbol):

    stock = yf.Ticker(symbol)
    info = stock.info

    data = {
        "company_name": info.get("longName"),
        "business_summary": info.get("longBusinessSummary"),
        "industry": info.get("industry"),
        "sector": info.get("sector"),
        "key_officers": info.get("companyOfficers")
    }

    return jsonify(data)


# Stock Market Data Endpoint
@app.route("/stock-market-data/<symbol>", methods=["GET"])
def stock_market_data(symbol):

    stock = yf.Ticker(symbol)
    info = stock.info

    current_price = info.get("currentPrice")
    previous_close = info.get("previousClose")

    data = {
        "symbol": symbol.upper(),
        "market_state": info.get("marketState"),
        "current_price": current_price,
        "previous_close": previous_close,
        "open": info.get("open"),
        "day_high": info.get("dayHigh"),
        "day_low": info.get("dayLow"),
        "price_change": (
            current_price - previous_close
            if current_price and previous_close else None
        ),
        "percentage_change": (
            ((current_price - previous_close) / previous_close) * 100
            if current_price and previous_close else None
        )
    }

    return jsonify(data)


# Historical Data Endpoint
@app.route("/historical-data", methods=["POST"])
def historical_data():

    data = request.get_json()

    symbol = data.get("symbol")
    start_date = data.get("start_date")
    end_date = data.get("end_date")

    stock = yf.Ticker(symbol)

    hist = stock.history(start=start_date, end=end_date)

    result = hist.reset_index().to_dict(orient="records")

    return jsonify(result)


# Analytical Insights Endpoint
@app.route("/analytical-insights", methods=["POST"])
def analytical_insights():

    data = request.get_json()

    symbol = data.get("symbol")
    start_date = data.get("start_date")
    end_date = data.get("end_date")

    stock = yf.Ticker(symbol)

    hist = stock.history(start=start_date, end=end_date)

    if hist.empty:
        return jsonify({"error": "No historical data found"})

    insights = {
        "symbol": symbol.upper(),
        "highest_price": hist["High"].max(),
        "lowest_price": hist["Low"].min(),
        "average_close_price": hist["Close"].mean(),
        "total_volume": int(hist["Volume"].sum()),
        "price_change": hist["Close"].iloc[-1] - hist["Close"].iloc[0],
        "percentage_change": (
            ((hist["Close"].iloc[-1] - hist["Close"].iloc[0])
             / hist["Close"].iloc[0]) * 100
        )
    }

    return jsonify(insights)


# Run Flask app
if __name__ == "__main__":
    app.run(debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [19/May/2026 20:27:29] "GET /company-info/AAPL HTTP/1.1" 200 -
127.0.0.1 - - [19/May/2026 20:27:29] "GET /favicon.ico HTTP/1.1" 404 -


Output

In [ ]:
# http://127.0.0.1:5000/company-info/AAPL
# http://127.0.0.1:5000/stock-market-data/AAPL